# Laboratorio: chatbot con LangChain + Groq + Gradio
**Autor:** [Escribe tu nombre]  |  **Curso:** [Escribe el curso]

Este notebook sigue la estructura de tu Colab de referencia: instala `langchain-groq`, crea un `ChatPromptTemplate`, invoca un modelo Groq y publica una interfaz Gradio temporal. El selector de rol escoge un prompt distinto para cada consulta.

## 1. Instalar librerías

In [1]:
!pip -q install -U langchain langchain-groq gradio

## 2. Importar y configurar Groq
En Google Colab abre el panel **Secretos** (icono de llave), crea `GROQ_API_KEY` con tu clave y activa el acceso para este cuaderno. No pegues la clave directamente en una celda ni la compartas en capturas.

In [3]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY
)

print("Conexión configurada. La clave no se muestra.")

ModuleNotFoundError: No module named 'google.colab'

## 3. Crear un prompt para cada rol
`ChatPromptTemplate` recibe la consulta y se combina con el modelo en una cadena LangChain.

In [ ]:
prompts_por_rol = {
    "Médico": ChatPromptTemplate.from_template(
        """Actúa como orientador de salud y brinda información médica general y educativa, considerando principios y contexto de la OMS. Responde en español claro, explica medidas preventivas y señala cuándo buscar atención profesional. No diagnostiques ni prescribas tratamientos. Si la consulta describe una emergencia, recomienda acudir de inmediato a los servicios de emergencia locales. Aclara que tu respuesta no sustituye una valoración médica.

Consulta: {pregunta}
Respuesta:"""
    ),
    "Programador": ChatPromptTemplate.from_template(
        """Actúa como especialista en programación, código y arquitectura de sistemas. Responde en español con explicaciones claras, pasos concretos y ejemplos de código cuando ayuden. Menciona supuestos, riesgos y buenas prácticas. Si faltan datos como lenguaje, versión o entorno, indícalo o pregunta. No inventes APIs ni afirmes haber ejecutado código.

Consulta: {pregunta}
Respuesta:"""
    ),
    "Abogado": ChatPromptTemplate.from_template(
        """Actúa como asistente de orientación jurídica general considerando el contexto de Guatemala. Explica conceptos y opciones en lenguaje sencillo. No presentes la respuesta como asesoría legal personalizada, no garantices resultados y recomienda consultar a un abogado colegiado para decisiones concretas. No inventes artículos de ley; si no tienes certeza de una norma vigente, indícalo y recomienda verificarla en una fuente oficial. Responde en español.

Consulta: {pregunta}
Respuesta:"""
    ),
}

print("Roles listos:", ", ".join(prompts_por_rol.keys()))

## 4. Función que escoge el prompt según el selector

In [ ]:
def responder(rol, pregunta):
    if not pregunta or not pregunta.strip():
        return "Escribe una pregunta para recibir una respuesta."
    prompt = prompts_por_rol[rol]
    cadena = prompt | llm
    resultado = cadena.invoke({"pregunta": pregunta.strip()})
    return resultado.content

## 5. Crear la interfaz de Gradio
Al ejecutar esta celda, Gradio muestra un enlace temporal `*.gradio.live`. Abre ese enlace para probar los tres perfiles. El enlace solo permanece activo mientras siga ejecutándose la sesión de Colab.

In [ ]:
import gradio as gr

interfaz = gr.Interface(
    fn=responder,
    inputs=[
        gr.Dropdown(
            choices=list(prompts_por_rol.keys()),
            value="Programador",
            label="Selecciona un rol",
        ),
        gr.Textbox(
            lines=4,
            label="Escribe tu pregunta",
            placeholder="Escribe aquí tu consulta…",
        ),
    ],
    outputs=gr.Textbox(lines=10, label="Respuesta"),
    title="Chatbot de roles",
    description="Elige un rol y recibe una respuesta orientada por su prompt especializado.",
)
interfaz.launch(share=True, debug=True)

## 6. Pruebas y entrega
Prueba cada perfil con una consulta distinta:
- **Médico:** ¿Qué medidas generales ayudan a prevenir la deshidratación?
- **Programador:** ¿Qué diferencia hay entre una API REST y una API GraphQL?
- **Abogado:** ¿Qué pasos generales puedo seguir ante un conflicto laboral en Guatemala?

Toma una captura por rol donde se vea el selector, la pregunta y la respuesta. No incluyas la clave API. Descarga el notebook desde **Archivo → Descargar → Descargar .ipynb** y agrega las capturas a un documento para exportarlo a PDF.

**Nota:** Las respuestas médicas y jurídicas son información general; verifica asuntos concretos con profesionales y fuentes oficiales.